# Qdrant Semantic Search

Cleaned Google Colab notebook.

## Cell 1: Install dependencies and load the embedding model

In [ ]:
%pip install -q qdrant-client sentence-transformers

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
DIM = 384

## Cell 2: Prepare the sample text corpus

In [ ]:
facts = [
    "Octopuses have three hearts and blue blood.",
    "The Eiffel Tower can grow taller in summer due to heat expansion.",
    "Bananas are technically berries, but strawberries are not.",
    "Honey never spoils; 3000-year-old honey was found in Egyptian tombs.",
    "A day on Venus is longer than a year on Venus.",
    "Sharks existed before trees did.",
    "The inventor of the Pringles can is buried in one.",
    "Cows have best friends and get stressed when separated.",
    "The unicorn is the national animal of Scotland.",
    "Wombat poop is cube-shaped.",
    "There are more possible chess games than atoms in the observable universe.",
    "The Great Wall of China is not visible from space with the naked eye.",
    "A group of flamingos is called a flamboyance.",
    "The shortest war in history lasted 38 minutes.",
    "Sea otters hold hands while sleeping to avoid drifting apart.",
    "The mitochondria in your cells have their own DNA.",
    "Lightning strikes Earth about 100 times every second.",
    "The Sahara Desert was green 6000 years ago.",
    "A teaspoon of neutron star weighs about a billion tons.",
    "Penguins propose to their mates with pebbles.",
    "The first computer bug was an actual moth.",
    "Cats cannot taste sweetness.",
    "The Pacific Ocean is shrinking by a few centimeters per year.",
    "A bolt of lightning is hotter than the surface of the sun.",
    "Butterflies taste with their feet.",
    "The dot over a lowercase 'i' is called a tittle.",
    "Sound travels about 4 times faster in water than in air.",
    "Turtles can breathe through their butts.",
    "The Beatles' 'Hey Jude' was originally called 'Hey Jules'.",
    "A snail can sleep for three years.",
]

## Cell 3: Create the Qdrant collection and add embedded facts

In [ ]:
vectors = model.encode(facts, normalize_embeddings=True)

client = QdrantClient(":memory:")
client.create_collection(
    collection_name="random_facts",
    vectors_config=VectorParams(size=DIM, distance=Distance.COSINE),
)

client.upsert(
    collection_name="random_facts",
    points=[
        PointStruct(
            id=i,
            vector=vectors[i].tolist(),
            payload={"text": facts[i]},
        )
        for i in range(len(facts))
    ],
)

def search(query, k=3):
    query_vector = model.encode(query, normalize_embeddings=True).tolist()
    results = client.query_points(
        collection_name="random_facts",
        query=query_vector,
        limit=k,
    ).points

    print(f"\nQuery: {query}")
    for result in results:
        print(f" {result.score:.3f} {result.payload['text']}")

## Cell 4: Run example semantic searches

In [ ]:
search("Which animal has multiple hearts?")
search("Things about space and the sun")
search("Weird animal sleep habits")
search("What did the Beatles originally call a song?")